# Exercice 5 - Portefeuille homogène et non homogène

In [ ]:
import numpy as np
from scipy.stats import norm, multivariate_normal
from scipy.integrate import quad
import matplotlib.pyplot as plt

## Partie A - Portefeuille homogène granulaire

### Question 1 : Moyenne et variance

Quand N tend vers l'infini (portefeuille granulaire), par la loi des grands nombres conditionnellement à F, la perte du portefeuille converge vers :

L|F = LGD * p(F)

où p(F) est la proba de defaut conditionnelle au facteur.

La moyenne c'est simplement E[L] = LGD * PD.

Pour la variance, il faut passer par la probabilité jointe de défaut de deux noms (normale bivariée) :

Var(L) = LGD² * (Phi_2(s, s, rho) - PD²)

avec s = Phi^{-1}(PD).

In [ ]:
PD = 0.02
LGD = 0.45
rho = 0.15

In [ ]:
s = norm.ppf(PD)

EL = LGD * PD
print(f"E[L] = {EL*100:.3f}%")

# variance via normale bivariée
cov = [[1, rho], [rho, 1]]
joint = multivariate_normal.cdf([s, s], mean=[0,0], cov=cov)
var_L = LGD**2 * (joint - PD**2)
print(f"Var(L) = {var_L:.6f}")
print(f"Ecart-type = {np.sqrt(var_L)*100:.3f}%")

### Question 2 : VaR

La formule de Vasicek donne directement le quantile de la perte. C'est la fameuse formule qu'on retrouve dans Bâle II. On inverse la relation entre F et L pour trouver le quantile alpha.

In [ ]:
def VaR_vasicek(alpha, PD, LGD, rho):
    return LGD * norm.cdf((norm.ppf(PD) + np.sqrt(rho)*norm.ppf(alpha)) / np.sqrt(1-rho))

In [ ]:
for alpha in [0.95, 0.99, 0.995, 0.999]:
    v = VaR_vasicek(alpha, PD, LGD, rho)
    print(f"VaR({alpha*100:.1f}%) = {v*100:.3f}%  |  Capital éco = {(v-EL)*100:.3f}%")

### Question 3 : E[(L - l0)+]

C'est l'expected loss d'une tranche senior avec point d'attachement l0. On calcule par intégration numérique sur le facteur F.

In [ ]:
def EL_senior(l0, PD, LGD, rho):
    def integ(f):
        L = LGD * norm.cdf((norm.ppf(PD) - np.sqrt(rho)*f) / np.sqrt(1-rho))
        return max(L - l0, 0) * norm.pdf(f)
    res, _ = quad(integ, -6, 6)
    return res

In [ ]:
l0_vals = np.linspace(0, LGD*0.7, 25)
el_vals = [EL_senior(l0, PD, LGD, rho) for l0 in l0_vals]

plt.figure(figsize=(7, 4))
plt.plot(l0_vals*100, [e*100 for e in el_vals], 'b-', lw=2)
plt.xlabel('Seuil l0 (%)')
plt.ylabel('E[(L-l0)+] (%)')
plt.title('EL tranche senior vs attachement')
plt.grid(alpha=0.3)
plt.show()

## Partie B - Portefeuille non homogène

### Question 1

Ici chaque actif a sa propre proba de defaut (terme sigma * eps'_i). Le defaut a lieu si :

rho*F + sqrt(1-rho)*eps_i <= s + sigma*eps'_i

En réarrangeant : rho*F + sqrt(1-rho)*eps_i - sigma*eps'_i <= s

Le membre de gauche est gaussien. La partie idiosyncratique (sqrt(1-rho)*eps - sigma*eps') a une variance de (1-rho) + sigma². La variance totale est rho² + (1-rho) + sigma².

En normalisant on retombe sur un modèle Vasicek avec :
- rho* = rho² / (rho² + 1 - rho + sigma²)
- s* = s / sqrt(rho² + 1 - rho + sigma²)
- PD* = Phi(s*)

In [ ]:
sigma = 0.20

sig_tot2 = rho**2 + (1-rho) + sigma**2
rho_star = rho**2 / sig_tot2
s_star = s / np.sqrt(sig_tot2)
PD_star = norm.cdf(s_star)

print(f"Parametres originaux : PD={PD}, rho={rho}")
print(f"Parametres modifiés  : PD*={PD_star:.4f}, rho*={rho_star:.4f}")

In [ ]:
v1 = VaR_vasicek(0.999, PD, LGD, rho)
v2 = VaR_vasicek(0.999, PD_star, LGD, rho_star)
print(f"VaR 99.9% homogene   = {v1*100:.3f}%")
print(f"VaR 99.9% heterogene = {v2*100:.3f}%")
print("\nL'heterogeneité reduit rho effectif donc le risque systémique (queue plus fine).")